In [1]:
# !pip install chromadb rank_bm25 
# !pip install sentence-transformers

In [2]:
import chromadb
from rank_bm25 import BM25Okapi
import re

In [3]:
documents = [
    "The Eiffel Tower is located in Paris, France and was completed in 1889.",
    "Python is a high-level programming language known for readability.",
    "Machine learning models require large amounts of training data.",
    "The Great Wall of China stretches over 13,000 miles.",
    "Neural networks are inspired by the structure of the human brain.",
    "Paris is the capital city of France and a major tourist destination.",
    "BM25 is a ranking function used by search engines like Elasticsearch.",
    "Deep learning is a subset of machine learning using neural networks.",
    "The Louvre Museum in Paris houses the Mona Lisa painting.",
    "Vector databases store embeddings for efficient similarity search.",
]

doc_ids = [f"doc_{i}" for i in range(len(documents))]

# chroma setup (basic)

In [4]:
client = chromadb.Client()
collection = client.create_collection(name="teaching_demo")

collection.add(
    documents=documents,
    ids=doc_ids
)

def vector_search(query, k=5):
    results = collection.query(query_texts=[query], n_results=k)
    return list(zip(results["ids"][0], results["documents"][0], results["distances"][0]))

query = "landmarks in France"
vector_results = vector_search(query)
for doc_id, doc, dist in vector_results:
    print(f"{doc_id} | dist={dist:.4f} | {doc}")

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event CollectionAddEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


doc_5 | dist=0.7526 | Paris is the capital city of France and a major tourist destination.
doc_0 | dist=1.1582 | The Eiffel Tower is located in Paris, France and was completed in 1889.
doc_8 | dist=1.2259 | The Louvre Museum in Paris houses the Mona Lisa painting.
doc_9 | dist=1.6467 | Vector databases store embeddings for efficient similarity search.
doc_3 | dist=1.6612 | The Great Wall of China stretches over 13,000 miles.


# bm25 setup

In [5]:
def tokenize(text):
    return re.findall(r"\w+", text.lower())

tokenized_corpus = [tokenize(doc) for doc in documents]
bm25 = BM25Okapi(tokenized_corpus)

def bm25_search(query, k=5):
    tokenized_query = tokenize(query)
    scores = bm25.get_scores(tokenized_query)
    ranked = sorted(zip(doc_ids, documents, scores), key=lambda x: x[2], reverse=True)
    return ranked[:k]

bm25_results = bm25_search(query)
for doc_id, doc, score in bm25_results:
    print(f"{doc_id} | score={score:.4f} | {doc}")

doc_0 | score=2.7293 | The Eiffel Tower is located in Paris, France and was completed in 1889.
doc_8 | score=1.2506 | The Louvre Museum in Paris houses the Mona Lisa painting.
doc_5 | score=1.1499 | Paris is the capital city of France and a major tourist destination.
doc_1 | score=0.0000 | Python is a high-level programming language known for readability.
doc_2 | score=0.0000 | Machine learning models require large amounts of training data.


In [6]:
print("Query:", query)
print("\nVector search ranking:")
for i, (doc_id, doc, _) in enumerate(vector_results):
    print(f"  {i+1}. {doc_id}: {doc}")

print("\nBM25 ranking:")
for i, (doc_id, doc, _) in enumerate(bm25_results):
    print(f"  {i+1}. {doc_id}: {doc}")

Query: landmarks in France

Vector search ranking:
  1. doc_5: Paris is the capital city of France and a major tourist destination.
  2. doc_0: The Eiffel Tower is located in Paris, France and was completed in 1889.
  3. doc_8: The Louvre Museum in Paris houses the Mona Lisa painting.
  4. doc_9: Vector databases store embeddings for efficient similarity search.
  5. doc_3: The Great Wall of China stretches over 13,000 miles.

BM25 ranking:
  1. doc_0: The Eiffel Tower is located in Paris, France and was completed in 1889.
  2. doc_8: The Louvre Museum in Paris houses the Mona Lisa painting.
  3. doc_5: Paris is the capital city of France and a major tourist destination.
  4. doc_1: Python is a high-level programming language known for readability.
  5. doc_2: Machine learning models require large amounts of training data.


# RRF

In [7]:
def reciprocal_rank_fusion(result_lists, k=60):
    """
    result_lists: list of ranked lists, each a list of doc_ids in rank order
    k: constant that dampens the impact of high ranks (standard default = 60)
    """
    fused_scores = {}
    for result_list in result_lists:
        for rank, doc_id in enumerate(result_list):
            fused_scores.setdefault(doc_id, 0)
            fused_scores[doc_id] += 1 / (k + rank + 1)

    reranked = sorted(fused_scores.items(), key=lambda x: x[1], reverse=True)
    return reranked

vector_ids = [doc_id for doc_id, _, _ in vector_results]
bm25_ids = [doc_id for doc_id, _, _ in bm25_results]

fused = reciprocal_rank_fusion([vector_ids, bm25_ids])

id_to_doc = dict(zip(doc_ids, documents))
print("Fused ranking (RRF):")
for doc_id, score in fused:
    print(f"  {doc_id} | rrf_score={score:.4f} | {id_to_doc[doc_id]}")

Fused ranking (RRF):
  doc_0 | rrf_score=0.0325 | The Eiffel Tower is located in Paris, France and was completed in 1889.
  doc_5 | rrf_score=0.0323 | Paris is the capital city of France and a major tourist destination.
  doc_8 | rrf_score=0.0320 | The Louvre Museum in Paris houses the Mona Lisa painting.
  doc_9 | rrf_score=0.0156 | Vector databases store embeddings for efficient similarity search.
  doc_1 | rrf_score=0.0156 | Python is a high-level programming language known for readability.
  doc_3 | rrf_score=0.0154 | The Great Wall of China stretches over 13,000 miles.
  doc_2 | rrf_score=0.0154 | Machine learning models require large amounts of training data.


In [8]:
def hybrid_search(query, k=5, rrf_k=60):
    v_results = vector_search(query, k=k)
    b_results = bm25_search(query, k=k)

    v_ids = [doc_id for doc_id, _, _ in v_results]
    b_ids = [doc_id for doc_id, _, _ in b_results]

    fused = reciprocal_rank_fusion([v_ids, b_ids], k=rrf_k)
    return [(doc_id, id_to_doc[doc_id], score) for doc_id, score in fused]

for doc_id, doc, score in hybrid_search("machine learning and neural networks"):
    print(f"{doc_id} | {score:.4f} | {doc}")

doc_7 | 0.0328 | Deep learning is a subset of machine learning using neural networks.
doc_4 | 0.0320 | Neural networks are inspired by the structure of the human brain.
doc_2 | 0.0320 | Machine learning models require large amounts of training data.
doc_9 | 0.0156 | Vector databases store embeddings for efficient similarity search.
doc_5 | 0.0156 | Paris is the capital city of France and a major tourist destination.
doc_6 | 0.0154 | BM25 is a ranking function used by search engines like Elasticsearch.
doc_0 | 0.0154 | The Eiffel Tower is located in Paris, France and was completed in 1889.


# Cross Encoder Re-ranker

In [9]:
# !pip install sentence-transformers

from sentence_transformers import CrossEncoder

# Fast, lightweight — good for live demos
reranker_fast = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

# Stronger quality, still open-source and self-hostable
reranker_strong = CrossEncoder("BAAI/bge-reranker-v2-m3")

def rerank(query, candidates, model, top_k=3):
    pairs = [(query, doc) for _, doc, _ in candidates]
    scores = model.predict(pairs)
    reranked = sorted(zip(candidates, scores), key=lambda x: x[1], reverse=True)
    return [(doc_id, doc, score) for ((doc_id, doc, _), score) in reranked[:top_k]]

In [10]:
query = "machine learning and neural networks"
candidates = hybrid_search(query, k=8)

print("MiniLM reranker")
for doc_id, doc, score in rerank(query, candidates, reranker_fast, top_k=3):
    print(f"{doc_id} | {score:.4f} | {doc}")

print("\nBGE reranker v2-m3")
for doc_id, doc, score in rerank(query, candidates, reranker_strong, top_k=3):
    print(f"{doc_id} | {score:.4f} | {doc}")

MiniLM reranker
doc_7 | 6.4655 | Deep learning is a subset of machine learning using neural networks.
doc_4 | 1.1385 | Neural networks are inspired by the structure of the human brain.
doc_2 | -1.2269 | Machine learning models require large amounts of training data.

BGE reranker v2-m3
doc_7 | 0.9648 | Deep learning is a subset of machine learning using neural networks.
doc_2 | 0.0781 | Machine learning models require large amounts of training data.
doc_4 | 0.0261 | Neural networks are inspired by the structure of the human brain.


# testing current pipeline

In [11]:
# !pip install beir

In [12]:
from beir import util
from beir.datasets.data_loader import GenericDataLoader
import pathlib, os

dataset_name = "scifact"
url = f"https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/{dataset_name}.zip"

out_dir = os.path.join(os.getcwd(), "datasets")
data_path = util.download_and_unzip(url, out_dir)

corpus, queries, qrels = GenericDataLoader(data_folder=data_path).load(split="test")

print(f"Corpus size: {len(corpus)}")
print(f"Number of queries: {len(queries)}")
print(f"Number of qrels: {len(qrels)}")

print(corpus["4983"]  ) # -> {"title": "...", "text": "..."}
print(queries["36"])     # -> "some question string"
print(qrels["36"]    )    # -> {"4983": 1, "8271": 2}   (doc_id -> relevance grade)

  0%|          | 0/5183 [00:00<?, ?it/s]

Corpus size: 5183
Number of queries: 300
Number of qrels: 300
{'text': 'Alterations of the architecture of cerebral white matter in the developing human brain can affect cortical development and result in functional disabilities. A line scan diffusion-weighted magnetic resonance imaging (MRI) sequence with diffusion tensor analysis was applied to measure the apparent diffusion coefficient, to calculate relative anisotropy, and to delineate three-dimensional fiber architecture in cerebral white matter in preterm (n = 17) and full-term infants (n = 7). To assess effects of prematurity on cerebral white matter development, early gestation preterm infants (n = 10) were studied a second time at term. In the central white matter the mean apparent diffusion coefficient at 28 wk was high, 1.8 microm2/ms, and decreased toward term to 1.2 microm2/ms. In the posterior limb of the internal capsule, the mean apparent diffusion coefficients at both times were similar (1.2 versus 1.1 microm2/ms). Rel

In [13]:
list(qrels.keys())[:5]

['1', '3', '5', '13', '36']

In [14]:
import itertools

#using the full corpus now
subset_doc_ids = list(itertools.islice(corpus.keys(), 5183))  # keep it small for speed
demo_corpus = {doc_id: corpus[doc_id] for doc_id in subset_doc_ids}

demo_documents = [f"{v['title']} {v['text']}" for v in demo_corpus.values()]
demo_doc_ids = list(demo_corpus.keys())

print(demo_documents[0][:300])

Microstructural development of human newborn cerebral white matter assessed in vivo by diffusion tensor magnetic resonance imaging. Alterations of the architecture of cerebral white matter in the developing human brain can affect cortical development and result in functional disabilities. A line sca


In [15]:
demo_collection = client.create_collection(name="scifact_demo_1")
demo_collection.add(documents=demo_documents, ids=demo_doc_ids)

demo_tokenized = [tokenize(doc) for doc in demo_documents]
demo_bm25 = BM25Okapi(demo_tokenized)

demo_id_to_doc = dict(zip(demo_doc_ids, demo_documents))

Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event CollectionAddEvent: capture() takes 1 positional argument but 3 were given


In [16]:
# Pick a query that actually has relevance judgments in our subset
sample_qid = next(qid for qid in qrels if any(doc_id in demo_doc_ids for doc_id in qrels[qid]))
sample_query = queries[sample_qid]

print("Query:", sample_query)
print("Known relevant docs (qrels):", qrels[sample_qid])

Query: 0-dimensional biomaterials show inductive properties.
Known relevant docs (qrels): {'31715818': 1}


In [17]:
def vector_search(query, collection, id_to_doc, k=5):
    results = collection.query(query_texts=[query], n_results=k)
    return list(zip(results["ids"][0], results["documents"][0], results["distances"][0]))

In [18]:
def bm25_search(query, bm25_index, doc_ids, id_to_doc, k=5):
    tokenized_query = tokenize(query)
    scores = bm25_index.get_scores(tokenized_query)
    ranked = sorted(zip(doc_ids, [id_to_doc[d] for d in doc_ids], scores), key=lambda x: x[2], reverse=True)
    return ranked[:k]

In [19]:
def run_pipeline_on_queries(query_dict, collection, bm25_index, doc_ids, id_to_doc, k=10):
    results = {}
    for qid, qtext in query_dict.items():
        v_results = vector_search(qtext, collection, id_to_doc, k=k)
        b_results = bm25_search(qtext, bm25_index, doc_ids, id_to_doc, k=k)
        fused = reciprocal_rank_fusion(
            [[d for d, _, _ in v_results], [d for d, _, _ in b_results]]
        )
        results[qid] = {doc_id: float(score) for doc_id, score in fused[:k]}
    return results

sample_queries = {qid: queries[qid] for qid in list(qrels.keys())[:20]}
retrieval_results = run_pipeline_on_queries(
    sample_queries, demo_collection, demo_bm25, demo_doc_ids, demo_id_to_doc
)

from beir.retrieval.evaluation import EvaluateRetrieval
evaluator = EvaluateRetrieval()
ndcg, _map, recall, precision = evaluator.evaluate(qrels, retrieval_results, [10])

print("NDCG@10:", ndcg)
print("Recall@10:", recall)
print("precision:", precision)

Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


NDCG@10: {'NDCG@10': 0.74838}
Recall@10: {'Recall@10': 0.95}
precision: {'P@10': 0.105}


# eval with re ranking

In [20]:
def run_pipeline_on_queries(query_dict, collection, bm25_index, doc_ids, id_to_doc,
                             reranker_model=None, retrieve_k=20, final_k=10):
    results = {}
    for qid, qtext in query_dict.items():
        v_results = vector_search(qtext, collection, id_to_doc, k=retrieve_k)
        b_results = bm25_search(qtext, bm25_index, doc_ids, id_to_doc, k=retrieve_k)
        fused = reciprocal_rank_fusion(
            [[d for d, _, _ in v_results], [d for d, _, _ in b_results]]
        )
        # fused is [(doc_id, rrf_score), ...] -- reshape into (doc_id, doc_text, score) for rerank()
        candidates = [(doc_id, id_to_doc[doc_id], score) for doc_id, score in fused[:retrieve_k]]

        if reranker_model is not None:
            reranked = rerank(qtext, candidates, reranker_model, top_k=final_k)
            results[qid] = {doc_id: float(score) for doc_id, doc, score in reranked}
        else:
            results[qid] = {doc_id: float(score) for doc_id, score in fused[:final_k]}

    return results

In [ ]:
sample_queries = {qid: queries[qid] for qid in list(qrels.keys())[:20]} #using subset for testing purpose

# 1. Hybrid retrieval only (what we had before)
results_hybrid = run_pipeline_on_queries(
    sample_queries, demo_collection, demo_bm25, demo_doc_ids, demo_id_to_doc,
    reranker_model=None
)

# 2. Hybrid + MiniLM reranker
results_minilm = run_pipeline_on_queries(
    sample_queries, demo_collection, demo_bm25, demo_doc_ids, demo_id_to_doc,
    reranker_model=reranker_fast
)

# 3. Hybrid + BGE reranker
results_bge = run_pipeline_on_queries(
    sample_queries, demo_collection, demo_bm25, demo_doc_ids, demo_id_to_doc,
    reranker_model=reranker_strong
)

evaluator = EvaluateRetrieval()

for name, res in [("Hybrid only", results_hybrid),
                   ("Hybrid + MiniLM rerank", results_minilm),
                   ("Hybrid + BGE rerank", results_bge)]:
    ndcg, _map, recall, precision = evaluator.evaluate(qrels, res, [10])
    print(f"{name}: {ndcg}")